In [ ]:
# =================================================================
# SECTION 0: Setup and Imports with Confirmed File Path
# =================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
from pathlib import Path  # <-- Path is used for robust, OS-independent path handling

# --- 1. Define Model Parameters ---
TARGET_COLUMN = 'OT' # Assuming 'Oil Temperature' is the main variable to predict
T_IN = 96   # Encoder Lookback Window (e.g., 96 hours = 4 days)
T_OUT = 24  # Decoder Forecast Horizon (e.g., next 24 hours)

# --- 2. Define and Verify File Path ---
# Set the path based on your confirmed structure: Advanced-TS-Attention-Project/Data/ETTh1.csv
FILE_PATH = Path('Data') / 'ETTh1.csv' 

if not FILE_PATH.exists():
    # If the file isn't found, raise a specific error message
    raise FileNotFoundError(
        f"ERROR: Dataset not found! Please ensure 'ETTh1.csv' is located in the '{FILE_PATH.parent}/' directory."
    )
else:
    print(f"File found and confirmed at: {FILE_PATH}")

print(f"Lookback (T_in): {T_IN}, Forecast Horizon (T_out): {T_OUT}")

In [ ]:
# Load the dataset
df = pd.read_csv(FILE_PATH, parse_dates=['date'])
df.set_index('date', inplace=True)

print("Initial DataFrame Head:")
print(df.head())

# Check for missing values (if any, use interpolation or fillna)
print("\nMissing values check:")
print(df.isnull().sum())

# If missing values are found (e.g., in a real-world dataset):
# df.fillna(method='ffill', inplace=True)

In [ ]:
# --- A. Check for Non-Stationarity (ADF Test) ---
def check_stationarity(timeseries):
    print('Results of ADF Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','Lags Used','No. of Observations'])
    for key, value in dftest[4].items():
        dfoutput['Critical Value (%s)' % key] = value
    print(dfoutput)
    
    if dfoutput['p-value'] > 0.05:
        print("\nConclusion: Time Series is NON-STATIONARY (p > 0.05). Differencing is required.")
        return False
    else:
        print("\nConclusion: Time Series is STATIONARY (p <= 0.05). No differencing needed.")
        return True

# Check stationarity on the raw target data
check_stationarity(df[TARGET_COLUMN].dropna())

# --- B. Apply Differencing (if Non-Stationary) ---
# NOTE: Differencing is applied to the raw data BEFORE scaling.

if not check_stationarity(df[TARGET_COLUMN].dropna()):
    print("\nApplying first-order differencing to the target variable...")
    # Create a new column for the differenced target
    df['OT_diff'] = df[TARGET_COLUMN].diff()
    # Replace the original target column for subsequent steps (if needed) or create a new column set
    # For now, we will track the original and differenced columns for selective use.
    
    # Check stationarity again on the differenced data
    # check_stationarity(df['OT_diff'].dropna())

In [ ]:
# Section 3: Scaling (Normalization)

# Drop the first row if differencing was applied, as it contains a NaN
if 'OT_diff' in df.columns:
    df.dropna(inplace=True) 

# CRITICAL FIX: Add a second, unconditional dropna() to remove any NaN 
# from input features before scaling. This cleans the X_train arrays.
df.dropna(inplace=True) 

# Initialize scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Scale all features (excluding the differenced column if you plan to use the original target for final output)
features_to_scale = df.columns
# --- CRITICAL LINE: This creates df_scaled_np and df_scaled ---
df_scaled_np = scaler.fit_transform(df[features_to_scale])
df_scaled = pd.DataFrame(df_scaled_np, columns=features_to_scale, index=df.index)

# Isolate the scaler for the TARGET column only (needed for final evaluation)
target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler.fit(df[[TARGET_COLUMN]]) # Fit on the original, non-differenced target values

print("\nScaled DataFrame Head:")
print(df_scaled.head())

# The number of features (F)
F = df_scaled.shape[1]
print(f"\nNumber of Features (F): {F}")

In [ ]:
# --- Identify the target feature's index in the scaled DataFrame ---
# We need this index to extract the target sequence (Y)
target_feature_index = df_scaled.columns.get_loc(TARGET_COLUMN) # Uses the original target column name

def create_sequences(data_df, T_in, T_out, target_idx):
    """
    Creates Encoder Input (X) and Decoder Target (Y) sequences.
    
    X: (N, T_in, F) - All features for the lookback window.
    Y: (N, T_out, 1) - Only the target feature for the forecast horizon.
    """
    X, Y = [], []
    data_np = data_df.values # Use NumPy array for fast indexing

    # The loop stops T_in + T_out steps from the end
    for i in range(len(data_np) - T_in - T_out + 1):
        
        # 1. ENCODER INPUT (X): All features for the past T_in steps
        X_seq = data_np[i : i + T_in, :]
        X.append(X_seq)
        
        # 2. DECODER TARGET (Y): Only the target feature for the future T_out steps
        # Start at T_in (the first step *after* the encoder window ends)
        Y_seq = data_np[i + T_in : i + T_in + T_out, target_idx]
        Y.append(Y_seq)

    # Reshape Y to be (N, T_out, 1) to match Keras model output shape
    return np.array(X), np.array(Y).reshape(-1, T_out, 1)

# Apply the function
X_all, Y_all = create_sequences(df_scaled, T_IN, T_OUT, target_feature_index)

print(f"\nFinal Shape of Encoder Input (X_all): {X_all.shape}")
print(f"Final Shape of Decoder Target (Y_all): {Y_all.shape}")

In [ ]:
# Define split ratios
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

N = X_all.shape[0] # Total number of samples

# Calculate split indices chronologically
train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

# --- Split the data arrays ---
X_train, Y_train = X_all[:train_end], Y_all[:train_end]
X_val, Y_val = X_all[train_end:val_end], Y_all[train_end:val_end]
X_test, Y_test = X_all[val_end:], Y_all[val_end:]

# --- Decoder Input Placeholder (Needed for Keras API) ---
# The Decoder requires an input sequence, often a sequence of zeros matching the shape (N, T_out, F).
# During training, this can be ignored/simplified, but the Keras model definition requires it.
# We will use the same shape as X_train but with T_out timesteps.

# Create a placeholder of zeros for the Decoder Input
def create_decoder_input(N_samples, T_out, F):
    # This shape needs to match the decoder_inputs shape in the Keras model
    return np.zeros((N_samples, T_out, F))

decoder_input_train = create_decoder_input(X_train.shape[0], T_OUT, F)
decoder_input_val = create_decoder_input(X_val.shape[0], T_OUT, F)
decoder_input_test = create_decoder_input(X_test.shape[0], T_OUT, F)

print("\n--- Final Dataset Shapes ---")
print(f"X_train (Encoder Input): {X_train.shape}")
print(f"Y_train (Decoder Target): {Y_train.shape}")
print(f"Decoder_Input_train: {decoder_input_train.shape}")
print("-" * 30)
print(f"X_test (Test Encoder Input): {X_test.shape}")
print(f"Y_test (Test Decoder Target): {Y_test.shape}")

In [ ]:
# =================================================================
# SECTION 6: Save Data Arrays and Scaler (FINAL REVISION)
# =================================================================
import os
import numpy as np
import pickle

# --- USE YOUR FOLDER NAME HERE ---
RESULTS_FOLDER = 'Results' 

# Ensure the results directory exists
os.makedirs(RESULTS_FOLDER, exist_ok=True)

# Save the final NumPy arrays
print("Saving final NumPy arrays...")
np.save(f'{RESULTS_FOLDER}/X_train.npy', X_train)
np.save(f'{RESULTS_FOLDER}/Y_train.npy', Y_train)
np.save(f'{RESULTS_FOLDER}/X_val.npy', X_val)
np.save(f'{RESULTS_FOLDER}/Y_val.npy', Y_val)
np.save(f'{RESULTS_FOLDER}/decoder_input_train.npy', decoder_input_train)
np.save(f'{RESULTS_FOLDER}/decoder_input_val.npy', decoder_input_val)
np.save(f'{RESULTS_FOLDER}/X_test.npy', X_test)
np.save(f'{RESULTS_FOLDER}/Y_test.npy', Y_test)
np.save(f'{RESULTS_FOLDER}/decoder_input_test.npy', decoder_input_test)

# Save the target_scaler object for inverse transformation later (Task 3)
with open(f'{RESULTS_FOLDER}/target_scaler.pkl', 'wb') as file:
    pickle.dump(target_scaler, file)

print(f"All data arrays and scaler successfully saved to {RESULTS_FOLDER}/ folder.")

In [ ]:
# --- DIAGNOSTIC CODE ---
print("--- NaN Count Check ---")
print(df.isnull().sum())
print("-" * 30)